<a href="https://colab.research.google.com/github/to-nakanishi/home_credit_default_risk/blob/main/home_credit_default_risk_05_ANALYZE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#0-1　基本設定

import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import roc_auc_score
import pickle
import gc
import warnings
import time
import os

warnings.filterwarnings("ignore")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
#1-1　寄与度確認　LGBM結果復元

SAVE_DIR = "/content/drive/MyDrive/home_credit_models_gpu"

oof_lgbm_df = pd.read_parquet(os.path.join(SAVE_DIR, "oof_lgbm.parquet"))
oof_lgbm = oof_lgbm_df["oof_lgbm"].values
y = oof_lgbm_df["TARGET"].values

pred_lgbm_df = pd.read_parquet(os.path.join(SAVE_DIR, "pred_lgbm.parquet"))
preds_lgbm = pred_lgbm_df["pred_lgbm"].values
test_ids = pred_lgbm_df["SK_ID_CURR"].values

oof_auc_lgbm = roc_auc_score(y, oof_lgbm)
print(f"LGBM OOF AUC (復元): {oof_auc_lgbm:.5f}")

# ---  重要度データの復元 (LGBM) ---
lgb_imp_raw = pd.read_parquet(os.path.join(SAVE_DIR, "lgbm_importance.parquet"))
lgb_imp = lgb_imp_raw.groupby('feature')['importance'].mean().reset_index()
lgb_imp['LGBM_pct'] = (lgb_imp['importance'] / lgb_imp['importance'].sum()) * 100

LGBM OOF AUC (復元): 0.79271


In [ ]:
#1-2　寄与度確認　CAT結果復元

import glob

# CatBoost GPU版の復元
SAVE_DIR_CAT = "/content/drive/MyDrive/home_credit_models_gpu/home_credit_models_gpu_CAT"

oof_cb_df = pd.read_parquet(os.path.join(SAVE_DIR_CAT, "oof_catboost.parquet"))
oof_cb = oof_cb_df["oof_catboost"].values

pred_cb_df = pd.read_parquet(os.path.join(SAVE_DIR_CAT, "pred_catboost.parquet"))
preds_cb = pred_cb_df["pred_catboost"].values

oof_auc_cb = roc_auc_score(y, oof_cb)
print(f"CatBoost OOF AUC (GPU版復元): {oof_auc_cb:.5f}")

# ---  重要度データの復元 (CatBoost) ---
cat_imp_files = glob.glob(os.path.join(SAVE_DIR_CAT, "catboost_imp_fold*.parquet"))
if not cat_imp_files:
    print(f" {SAVE_DIR_CAT} 内にファイルが見つかりません。")
else:
    # 全Fold分を読み込んで平均化
    cat_imp_list = [pd.read_parquet(f) for f in cat_imp_files]
    cat_imp_raw = pd.concat(cat_imp_list)
    cat_imp = cat_imp_raw.groupby('feature')['importance'].mean().reset_index()

    # 合計を100%に正規化
    cat_imp['CAT_pct'] = (cat_imp['importance'] / cat_imp['importance'].sum()) * 100
    print(f"CatBoost重要度復元完了: {len(cat_imp)} 特徴量")

CatBoost OOF AUC (GPU版復元): 0.79233
CatBoost重要度復元完了: 669 特徴量


In [ ]:
#1-3　寄与度確認　アンサンブル（重み調整）

def find_best_weight(oof_a, oof_b, y_true, steps=101):
    """OOFスコアで最適なブレンド重みをグリッドサーチする。"""
    best_w, best_auc = 0.0, 0.0
    results = []
    for w in np.linspace(0, 1, steps):
        blended = w * oof_a + (1 - w) * oof_b
        auc = roc_auc_score(y_true, blended)
        results.append((w, auc))
        if auc > best_auc:
            best_w, best_auc = w, auc
    return best_w, best_auc, results

w_lgbm, ensemble_auc, weight_results = find_best_weight(oof_lgbm, oof_cb, y)

print("=" * 60)
print("アンサンブル結果")
print("=" * 60)
print(f"最適重み: LGBM={w_lgbm:.2f}, CatBoost={1 - w_lgbm:.2f}")
print(f"Ensemble OOF AUC: {ensemble_auc:.5f}")
print(f"  (LGBM単体: {oof_auc_lgbm:.5f})")
print(f"  (CatBoost単体: {oof_auc_cb:.5f})")

アンサンブル結果
最適重み: LGBM=0.59, CatBoost=0.41
Ensemble OOF AUC: 0.79411
  (LGBM単体: 0.79271)
  (CatBoost単体: 0.79233)


In [ ]:
#1-4　寄与度確認　LGBM,CAT

comparison_df = pd.merge(
        lgb_imp[['feature', 'LGBM_pct']],
        cat_imp[['feature', 'CAT_pct']],
        on='feature',
        how='outer'
    ).fillna(0)

# --- 4. 結果表示 ---
print(f"【モデル精度比較 (AUC)】")
print(f" - LightGBM: {oof_auc_lgbm:.5f}")
print(f" - CatBoost: {oof_auc_cb:.5f}")
print(f" - Ensemble: {ensemble_auc:.5f} (LGBM weight: {w_lgbm:.2f})\n")

# 各モデルのTOP15を抽出
lgb_top15 = comparison_df.sort_values(by='LGBM_pct', ascending=False).head(15)[['feature', 'LGBM_pct']]
cb_top15 = comparison_df.sort_values(by='CAT_pct', ascending=False).head(15)[['feature', 'CAT_pct']]

# 横に並べるための整形
lgb_top15 = lgb_top15.reset_index(drop=True).rename(columns={'feature': 'LGBM Feature', 'LGBM_pct': 'LGBM (%)'})
cb_top15 = cb_top15.reset_index(drop=True).rename(columns={'feature': 'CatBoost Feature', 'CAT_pct': 'CAT (%)'})
side_by_side = pd.concat([lgb_top15, cb_top15], axis=1)

print("■ モデル別重要度 TOP 15")
print(side_by_side.to_markdown(index=False, floatfmt=".2f"))

# --- 共通項目の抽出 ---
common_top = set(lgb_top15['LGBM Feature']) & set(cb_top15['CatBoost Feature'])
print(f"\n■ 両モデル共通の重要項目 ({len(common_top)}件):")
print(f"  {sorted(list(common_top))}")

【モデル精度比較 (AUC)】
 - LightGBM: 0.79271
 - CatBoost: 0.79233
 - Ensemble: 0.79411 (LGBM weight: 0.59)

■ モデル別重要度 TOP 15
| LGBM Feature                    |   LGBM (%) | CatBoost Feature                 |   CAT (%) |
|:--------------------------------|-----------:|:---------------------------------|----------:|
| EXT_SOURCES_GEOM_MEAN           |      11.24 | EXT_SOURCES_MEAN                 |      7.74 |
| EXT_SOURCES_MEAN                |       7.08 | EXT_SOURCES_GEOM_MEAN            |      5.31 |
| ORGANIZATION_TYPE               |       3.59 | CREDIT_ANNUITY_RATIO             |      3.04 |
| EXT_SOURCES_CUSTOM_MEAN         |       2.61 | DIFF_GOODS_CREDIT                |      1.61 |
| EXT_SOURCE_3                    |       2.54 | EXT_SOURCES_BIN_TARGET_RATE      |      1.26 |
| CREDIT_ANNUITY_RATIO            |       2.28 | INS_AMT_PAYMENT_min              |      1.17 |
| EXT_SOURCES_BIN_TARGET_RATE     |       1.66 | GEOM_MEAN_RISK                   |      1.13 |
| EXT_SOURCE_2     

In [ ]:
#1-5　寄与度確認　アンサンブル

# 各モデルの重要度にブレンド重みを掛ける
comparison_df['Ensemble_pct'] = (
    comparison_df['LGBM_pct'] * w_lgbm +
    comparison_df['CAT_pct'] * (1 - w_lgbm)
)

# --- 2. 結果の表示 ---

print(f"【アンサンブル寄与度分析】")
print(f" - 最適重み: LGBM={w_lgbm:.2f}, CatBoost={1-w_lgbm:.2f}")
print(f" - Ensemble OOF AUC: {ensemble_auc:.5f}\n")

# アンサンブル寄与度 TOP 20
ensemble_top20 = comparison_df.sort_values(by='Ensemble_pct', ascending=False).head(20)

print(f"■ アンサンブル全体での重要度 TOP 20 (436列中)")
print(ensemble_top20[['feature', 'Ensemble_pct', 'LGBM_pct', 'CAT_pct']].to_markdown(index=False, floatfmt=".2f"))

# --- 3. 考察用の統計 ---

# 片方のモデルでは無視されているが、アンサンブルで浮上した項目の確認
gap_features = comparison_df.assign(gap=abs(comparison_df['LGBM_pct'] - comparison_df['CAT_pct']))
print(f"\n■ モデル間で評価が大きく分かれた項目 (多様性の源泉):")
print(gap_features.sort_values(by='gap', ascending=False).head(5)[['feature', 'LGBM_pct', 'CAT_pct']].to_markdown(index=False))

【アンサンブル寄与度分析】
 - 最適重み: LGBM=0.59, CatBoost=0.41
 - Ensemble OOF AUC: 0.79411

■ アンサンブル全体での重要度 TOP 20 (436列中)
| feature                       |   Ensemble_pct |   LGBM_pct |   CAT_pct |
|:------------------------------|---------------:|-----------:|----------:|
| EXT_SOURCES_GEOM_MEAN         |           8.81 |      11.24 |      5.31 |
| EXT_SOURCES_MEAN              |           7.35 |       7.08 |      7.74 |
| CREDIT_ANNUITY_RATIO          |           2.59 |       2.28 |      3.04 |
| ORGANIZATION_TYPE             |           2.16 |       3.59 |      0.12 |
| EXT_SOURCES_CUSTOM_MEAN       |           1.94 |       2.61 |      0.97 |
| EXT_SOURCE_3                  |           1.87 |       2.54 |      0.90 |
| EXT_SOURCES_BIN_TARGET_RATE   |           1.49 |       1.66 |      1.26 |
| DIFF_GOODS_CREDIT             |           1.44 |       1.32 |      1.61 |
| EXT_SOURCE_2                  |           1.23 |       1.43 |      0.94 |
| GEOM_MEAN_RISK                |           1.12 |     